<a href="https://colab.research.google.com/github/fukagai-takuya/gifu-ai/blob/main/gifu-ai-2026-03-15/FLUX2_klein_4B_git_clone_from_fork_diffusers_check_image_latents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 勉強会用に fork した diffusers のリポジトリを git clone し、ディレクトリを diffusers に変えてから、調査用に用意したブランチ investigate_flux.2_klein_4B にブランチを変えます。このブランチには調査用のブレイクポイントが `Flux2KleinPipeline.__call__` の  Denoising loop の前にセットされています。

In [ ]:
!git clone https://github.com/fukagai-takuya/diffusers.git
%cd diffusers
!git checkout investigate_flux.2_klein_4B_image_latents

2. git clone した diffusers のソースコードを優先して参照するようにします

In [ ]:
import sys
sys.path.insert(0, "/content/diffusers/src")

3. git clone した diffusers のソースコードが優先して参照されることを確認します

In [ ]:
import diffusers
print(diffusers.__file__)

4. diffusers を使用して FLUX.2 [klein] 4B を実行する際に必要なパッケージをインストールします

In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers accelerate safetensors
!pip install sentencepiece protobuf einops

5. pdb より高機能な Python デバッガ ipdb をインストールします。(investigate_flux.2_klein_4B にセットしたのは、ipdb を使用するブレイクポイントなため)

In [ ]:
!pip install ipdb

6. FLUX.2 [klein] 4B を使用した Pipeline を用意します

In [ ]:
import torch
from diffusers import Flux2KleinPipeline

dtype = torch.bfloat16

pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=dtype)
pipe.enable_model_cpu_offload()  # save some VRAM by offloading the model to CPU


7. FLUX.2 [klein] 4B を使用し、プロンプト文字列を参照して画像を生成します

In [ ]:
device = "cuda"
prompt = 'A cat holding a sign that says "Gifu AI Study Group"'

image = pipe(
    prompt=prompt,
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=4,
    generator=torch.Generator(device=device).manual_seed(0)
).images[0]

image.save("flux-klein.png")

8. FLUX.2 [klein] 4B を使用し、入力画像とプロンプト文字列を参照して画像を生成します。下のコードでは、Web ページ上の指定した URL の画像を背景として、Gifu AI Study Group と書かれたボードを持った猫の画像を出力しようとしています。

In [ ]:
from diffusers.utils import load_image

device = "cuda"

url = 'https://www.leafwindow.com/wordpress-05/wp-content/uploads/2023/12/IMG_6492-20.jpg'
image = load_image(url)

prompt = """
Use the provided image as the background.
A cat holding a sign that says "Gifu AI Study Group" in the foreground,
realistic lighting, seamless integration, high detail
"""

image = pipe(
    prompt=prompt,
    image=image,
    height=1024,
    width=1024,
    guidance_scale=2.0,
    num_inference_steps=4,
    generator=torch.Generator(device=device).manual_seed(0)
).images[0]

image.save("flux-klein-with-input-image.png")